# Limpieza y preprocesamiento — NYC Yellow Taxi, mayo 2022

**Entrada:** `data/yellow_tripdata_2022-05.parquet`
**Salida:** `data/processed/train.parquet`, `val.parquet`, `test.parquet` (+ `limpieza_log.csv`)

| Split | Fechas (pickup) | Uso |
|---|---|---|
| Train | 1–21 may | Ajustar preprocesamiento y modelos |
| Validation | 22–26 may | Selección de features y tuning |
| Test | 27–31 may | Reporte final |

**Targets:** `fare_amount` (USD) y `trip_duration_min` (minutos) → métrica principal MAE.

**Features (filas amarillas):** `trip_distance`, `PULocationID`, `DOLocationID`, `pickup_hour`, `pickup_dow`, `is_airport_trip`, `morning_rush`, `night_rush`, `weekend_rush`.

**Regla de oro:** las reglas fijas (dominio) se aplican igual a todo; lo que se *aprende* de los datos (topes por cuantil) se calcula **solo con train** y luego se aplica a val y test.

## 1. Configuración

In [15]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# Raíz del proyecto = carpeta que contiene /data (sirve si el notebook está en la raíz o en /notebooks)
ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

RAW_PATH = ROOT / "data" / "yellow_tripdata_2022-05.parquet"
OUT_DIR = ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

GUARDAR_CSV = False  # True también guarda CSV (ojo: GitHub rechaza archivos > 100 MB)

# Split cronológico por fecha de recojo
INICIO_MES = pd.Timestamp("2022-05-01")
INICIO_VAL = pd.Timestamp("2022-05-22")
INICIO_TEST = pd.Timestamp("2022-05-27")
FIN_MES = pd.Timestamp("2022-06-01")

# Zonas TLC
ZONAS_AEROPUERTO = [1, 132, 138]   # Newark (EWR), JFK, LaGuardia
ZONAS_DESCONOCIDAS = [264, 265]    # Unknown / NA

# Ventanas de hora punta (ajustar si el equipo definió otras)
HORAS_MORNING_RUSH = list(range(7, 10))    # lun–vie 07:00–09:59
HORAS_NIGHT_RUSH = list(range(16, 20))     # lun–vie 16:00–19:59 (franja del recargo de hora punta TLC)
HORAS_WEEKEND_RUSH = list(range(10, 20))   # sáb–dom 10:00–19:59

# Reglas fijas de dominio (no se aprenden de los datos)
DURACION_MIN, DURACION_MAX = 1, 180        # minutos
DISTANCIA_MAX = 100                        # millas (además distancia > 0)
TARIFA_MIN, TARIFA_MAX = 2.5, 500          # USD; 2.50 = bajada de bandera en 2022
VELOCIDAD_MIN, VELOCIDAD_MAX = 1, 80       # mph
RATECODES_EXCLUIDOS = [5, 6, 99]           # tarifa negociada, viaje grupal, código inválido
PAYMENTS_EXCLUIDOS = [3, 4, 6]             # sin cobro, disputa, anulado

# Topes aprendidos SOLO con train
CUANTIL_TOPE = 0.999
COLS_TOPE = ["trip_distance", "trip_duration_min", "fare_amount"]

COLUMNAS_FEATURES = [
    "trip_distance", "PULocationID", "DOLocationID",
    "pickup_hour", "pickup_dow", "is_airport_trip",
    "morning_rush", "night_rush", "weekend_rush",
]
COLUMNAS_TARGET = ["fare_amount", "trip_duration_min"]
COLUMNAS_SALIDA = ["pickup_datetime"] + COLUMNAS_FEATURES + COLUMNAS_TARGET

print("ROOT    :", ROOT)
print("RAW_PATH:", RAW_PATH, "| existe:", RAW_PATH.exists())

ROOT    : c:\Users\FRANK FLORES\AnyoneAI_Proyect
RAW_PATH: c:\Users\FRANK FLORES\AnyoneAI_Proyect\data\yellow_tripdata_2022-05.parquet | existe: True


## 2. Carga de datos

In [16]:
raw = pd.read_parquet(RAW_PATH)
print(f"Filas: {len(raw):,} | Columnas: {raw.shape[1]}")
raw.info()
raw.head()

Filas: 3,588,295 | Columnas: 19
<class 'pandas.DataFrame'>
RangeIndex: 3588295 entries, 0 to 3588294
Data columns (total 19 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int64         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     str           
 7   PULocationID           int64         
 8   DOLocationID           int64         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  a

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,1,2022-05-01 00:00:36,2022-05-01 00:19:18,1.000,4.100,1.000,N,246,151,2,17.000,3.000,0.500,0.000,0.000,0.300,20.800,2.500,0.000
1,1,2022-05-01 00:27:44,2022-05-01 00:41:33,1.000,2.300,1.000,N,238,74,2,11.000,3.000,0.500,0.000,0.000,0.300,14.800,2.500,0.000
2,1,2022-05-01 00:59:00,2022-05-01 01:14:22,1.000,4.200,1.000,N,163,260,2,15.500,3.000,0.500,0.000,0.000,0.300,19.300,2.500,0.000
3,1,2022-05-01 00:48:18,2022-05-01 01:28:02,1.000,0.000,1.000,N,79,182,1,41.200,0.000,0.500,0.000,0.000,0.300,42.000,0.000,0.000
4,1,2022-05-01 00:28:26,2022-05-01 00:37:49,1.000,1.600,1.000,N,238,75,1,7.500,3.000,0.500,2.250,0.000,0.300,13.550,2.500,0.000


## 3. Diagnóstico de la data cruda

In [17]:
diag = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "nulos": raw.isna().sum(),
    "%_nulos": raw.isna().mean() * 100,
    "min": raw.min(numeric_only=True),
    "max": raw.max(numeric_only=True),
})
diag

,dtype,nulos,%_nulos,min,max
DOLocationID,int64,0,0.000,1.000,265.000
PULocationID,int64,0,0.000,1.000,265.000
RatecodeID,float64,129524,3.610,1.000,99.000
VendorID,int64,0,0.000,1.000,6.000
airport_fee,float64,129524,3.610,-1.250,1.250
congestion_surcharge,float64,129524,3.610,-2.500,2.750
extra,float64,0,0.000,-4.500,8.800
fare_amount,float64,0,0.000,"-1,311.500","6,966.500"
improvement_surcharge,float64,0,0.000,-0.300,0.300
mta_tax,float64,0,0.000,-0.500,3.300


In [18]:
dur_raw = (raw["tpep_dropoff_datetime"] - raw["tpep_pickup_datetime"]).dt.total_seconds() / 60
fuera_mes = ~raw["tpep_pickup_datetime"].between(INICIO_MES, FIN_MES, inclusive="left")

problemas = pd.Series({
    "pickup fuera de mayo 2022": fuera_mes.sum(),
    "duración <= 0 min": (dur_raw <= 0).sum(),
    "duración > 180 min": (dur_raw > DURACION_MAX).sum(),
    "trip_distance <= 0": (raw["trip_distance"] <= 0).sum(),
    "trip_distance > 100 mi": (raw["trip_distance"] > DISTANCIA_MAX).sum(),
    "fare_amount < 0": (raw["fare_amount"] < 0).sum(),
    "fare_amount < 2.50": (raw["fare_amount"] < TARIFA_MIN).sum(),
    "zona desconocida (264/265)": (raw["PULocationID"].isin(ZONAS_DESCONOCIDAS) | raw["DOLocationID"].isin(ZONAS_DESCONOCIDAS)).sum(),
    "filas duplicadas": raw.duplicated().sum(),
}, name="filas")
pd.DataFrame({"filas": problemas, "%": problemas / len(raw) * 100})

,filas,%
pickup fuera de mayo 2022,143,0.004
duración <= 0 min,3030,0.084
duración > 180 min,5084,0.142
trip_distance <= 0,46438,1.294
trip_distance > 100 mi,155,0.004
fare_amount < 0,20506,0.571
fare_amount < 2.50,22478,0.626
zona desconocida (264/265),65029,1.812
filas duplicadas,0,0.000


In [19]:
print("Fechas de pickup fuera de mayo (muestra):")
print(raw.loc[fuera_mes, "tpep_pickup_datetime"].dt.date.value_counts().head(10))
raw[["trip_distance", "fare_amount", "total_amount"]].assign(duracion_min=dur_raw).describe(
    percentiles=[0.001, 0.01, 0.5, 0.99, 0.999]
).T

Fechas de pickup fuera de mayo (muestra):
tpep_pickup_datetime
2022-04-30    106
2022-06-01     22
2008-12-31      8
2009-01-01      6
2003-01-01      1
Name: count, dtype: int64


,count,mean,std,min,0.1%,1%,50%,99%,99.9%,max
trip_distance,"3,588,295.000",6.857,690.849,0.000,0.000,0.000,1.960,20.300,29.700,"357,192.650"
fare_amount,"3,588,295.000",15.168,14.895,"-1,311.500",-22.000,2.500,10.500,63.000,119.000,"6,966.500"
total_amount,"3,588,295.000",22.078,18.487,"-1,314.800",-25.300,5.550,16.300,81.360,144.360,"6,970.800"
duracion_min,"3,588,295.000",18.218,51.567,-14.083,0.033,0.383,12.700,71.450,"1,397.528","6,823.550"


## 4. Limpieza con reglas fijas (dominio)

Se aplican a **todo el mes** porque no dependen de estadísticas de los datos. El log cuenta las filas eliminadas por cada regla **en orden** (cada regla solo cuenta lo que las anteriores no quitaron).

- `trip_duration_min` = dropoff − pickup (target en minutos).
- Coherencia tarifa–distancia/tiempo (taxímetro 2022: $2.50 bajada + $2.50/milla + $0.50/min lento): se elimina si `fare < 0.5·(2.5 + 2.5·d)` o `fare > 3·(2.5 + 2.5·d + 0.5·min) + 20`. Detecta distancias o tarifas mal registradas.
- Se quitan zonas 264/265 porque el equipo no conserva la bandera `unknow_zone`.

In [20]:
def limpiar_reglas_fijas(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Aplica reglas de dominio fijas. Devuelve (df_limpio, log de filas eliminadas por regla)."""
    df = df.copy()
    df["trip_duration_min"] = (
        df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
    ).dt.total_seconds() / 60

    d = df["trip_distance"]
    t = df["trip_duration_min"]
    fare = df["fare_amount"]
    velocidad_mph = d / (t / 60)
    tarifa_min_teorica = 2.5 + 2.5 * d
    tarifa_max_teorica = 2.5 + 2.5 * d + 0.5 * t

    reglas = {
        "pickup fuera de mayo 2022": ~df["tpep_pickup_datetime"].between(INICIO_MES, FIN_MES, inclusive="left"),
        "filas duplicadas": df.duplicated(),
        "zona desconocida (264/265)": df["PULocationID"].isin(ZONAS_DESCONOCIDAS) | df["DOLocationID"].isin(ZONAS_DESCONOCIDAS),
        "RatecodeID 5/6/99": df["RatecodeID"].isin(RATECODES_EXCLUIDOS),
        "payment_type 3/4/6": df["payment_type"].isin(PAYMENTS_EXCLUIDOS),
        f"duración fuera de [{DURACION_MIN}, {DURACION_MAX}] min": ~t.between(DURACION_MIN, DURACION_MAX),
        f"distancia fuera de (0, {DISTANCIA_MAX}] mi": ~((d > 0) & (d <= DISTANCIA_MAX)),
        f"tarifa fuera de [{TARIFA_MIN}, {TARIFA_MAX}] USD": ~fare.between(TARIFA_MIN, TARIFA_MAX),
        f"velocidad fuera de [{VELOCIDAD_MIN}, {VELOCIDAD_MAX}] mph": ~velocidad_mph.between(VELOCIDAD_MIN, VELOCIDAD_MAX),
        "tarifa incoherente con distancia/tiempo": (fare < 0.5 * tarifa_min_teorica) | (fare > 3 * tarifa_max_teorica + 20),
    }

    conservar = pd.Series(True, index=df.index)
    log = []
    for nombre, eliminar in reglas.items():
        eliminar = eliminar.fillna(True)
        n = int((eliminar & conservar).sum())
        log.append({"etapa": "fija", "regla": nombre, "filas_eliminadas": n})
        conservar &= ~eliminar

    log = pd.DataFrame(log)
    log["%_del_total"] = log["filas_eliminadas"] / len(df) * 100
    return df.loc[conservar].copy(), log


df, log_fijas = limpiar_reglas_fijas(raw)
n_raw = len(raw)
print(f"Antes: {n_raw:,} | Después: {len(df):,} | Eliminadas: {n_raw - len(df):,} ({(1 - len(df) / n_raw) * 100:.2f}%)")
log_fijas

Antes: 3,588,295 | Después: 3,420,641 | Eliminadas: 167,654 (4.67%)


,etapa,regla,filas_eliminadas,%_del_total
0,fija,pickup fuera de mayo 2022,143,0.004
1,fija,filas duplicadas,0,0.000
2,fija,zona desconocida (264/265),65026,1.812
3,fija,RatecodeID 5/6/99,28754,0.801
4,fija,payment_type 3/4/6,30381,0.847
5,fija,"duración fuera de [1, 180] min",24754,0.690
6,fija,"distancia fuera de (0, 100] mi",12241,0.341
7,fija,"tarifa fuera de [2.5, 500] USD",3592,0.100
8,fija,"velocidad fuera de [1, 80] mph",1694,0.047
9,fija,tarifa incoherente con distancia/tiempo,1069,0.030


In [21]:
del raw, dur_raw, fuera_mes  # libera memoria

## 5. Feature engineering

Features deterministas por fila (no aprenden nada de los datos → no hay fuga entre splits).
`pickup_dow`: 0 = lunes … 6 = domingo.

In [22]:
def crear_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    pickup = df["tpep_pickup_datetime"]
    df["pickup_hour"] = pickup.dt.hour.astype("int8")
    df["pickup_dow"] = pickup.dt.dayofweek.astype("int8")
    laborable = df["pickup_dow"] < 5

    df["is_airport_trip"] = (
        df["PULocationID"].isin(ZONAS_AEROPUERTO) | df["DOLocationID"].isin(ZONAS_AEROPUERTO)
    ).astype("int8")
    df["morning_rush"] = (laborable & df["pickup_hour"].isin(HORAS_MORNING_RUSH)).astype("int8")
    df["night_rush"] = (laborable & df["pickup_hour"].isin(HORAS_NIGHT_RUSH)).astype("int8")
    df["weekend_rush"] = (~laborable & df["pickup_hour"].isin(HORAS_WEEKEND_RUSH)).astype("int8")
    return df


df = crear_features(df)
df[["pickup_hour", "pickup_dow", "is_airport_trip", "morning_rush", "night_rush", "weekend_rush"]].describe().T

,count,mean,std,min,25%,50%,75%,max
pickup_hour,"3,420,641.000",14.190,5.771,0.000,10.000,15.000,19.000,23.000
pickup_dow,"3,420,641.000",2.918,1.999,0.000,1.000,3.000,5.000,6.000
is_airport_trip,"3,420,641.000",0.105,0.307,0.000,0.000,0.000,0.000,1.000
morning_rush,"3,420,641.000",0.091,0.288,0.000,0.000,0.000,0.000,1.000
night_rush,"3,420,641.000",0.196,0.397,0.000,0.000,0.000,0.000,1.000
weekend_rush,"3,420,641.000",0.159,0.366,0.000,0.000,0.000,0.000,1.000


## 6. Split cronológico

In [23]:
pickup = df["tpep_pickup_datetime"]
splits = {
    "train": df[pickup < INICIO_VAL],
    "val": df[(pickup >= INICIO_VAL) & (pickup < INICIO_TEST)],
    "test": df[pickup >= INICIO_TEST],
}
for nombre, d in splits.items():
    print(f"{nombre:>5}: {len(d):>10,} filas | {d['tpep_pickup_datetime'].min()} → {d['tpep_pickup_datetime'].max()}")

train:  2,399,190 filas | 2022-05-01 00:00:00 → 2022-05-21 23:59:59
  val:    548,505 filas | 2022-05-22 00:00:00 → 2022-05-26 23:59:58
 test:    472,946 filas | 2022-05-27 00:00:00 → 2022-05-31 23:59:59


## 7. Topes de outliers aprendidos SOLO con train

Se calcula el percentil 99.9 de distancia, duración y tarifa **en train** y se aplica el mismo tope a los tres splits.

In [24]:
topes = splits["train"][COLS_TOPE].quantile(CUANTIL_TOPE)
print(f"Topes (p{CUANTIL_TOPE * 100:g} de train):")
print(topes.round(2).to_string())

log_topes = []
for nombre in splits:
    d = splits[nombre]
    mantener = (d[COLS_TOPE] <= topes).all(axis=1)
    log_topes.append({
        "etapa": f"tope p{CUANTIL_TOPE * 100:g} (train)",
        "regla": f"{nombre}: supera algún tope",
        "filas_eliminadas": int((~mantener).sum()),
        "%_del_total": (~mantener).sum() / n_raw * 100,
    })
    splits[nombre] = d[mantener]

pd.DataFrame(log_topes)

Topes (p99.9 de train):
trip_distance       26.040
trip_duration_min   95.180
fare_amount         76.000


,etapa,regla,filas_eliminadas,%_del_total
0,tope p99.9 (train),train: supera algún tope,5999,0.167
1,tope p99.9 (train),val: supera algún tope,1743,0.049
2,tope p99.9 (train),test: supera algún tope,994,0.028


## 8. Columnas finales y tipos

In [25]:
def formatear_salida(d: pd.DataFrame) -> pd.DataFrame:
    out = d.rename(columns={"tpep_pickup_datetime": "pickup_datetime"})[COLUMNAS_SALIDA].copy()
    out["trip_duration_min"] = out["trip_duration_min"].round(2)
    out = out.astype({
        "trip_distance": "float32",
        "PULocationID": "int16",
        "DOLocationID": "int16",
        "fare_amount": "float32",
        "trip_duration_min": "float32",
    })
    return out.sort_values("pickup_datetime").reset_index(drop=True)


final = {nombre: formatear_salida(d) for nombre, d in splits.items()}
final["train"].head()

,pickup_datetime,trip_distance,PULocationID,DOLocationID,pickup_hour,pickup_dow,is_airport_trip,morning_rush,night_rush,weekend_rush,fare_amount,trip_duration_min
0,2022-05-01 00:00:00,8.230,246,42,0,6,0,0,0,0,37.340,27.000
1,2022-05-01 00:00:01,1.160,140,237,0,6,0,0,0,0,6.500,6.030
2,2022-05-01 00:00:01,0.550,140,140,0,6,0,0,0,0,3.500,1.950
3,2022-05-01 00:00:02,3.970,107,142,0,6,0,0,0,0,14.500,15.650
4,2022-05-01 00:00:02,1.540,161,137,0,6,0,0,0,0,9.000,11.200


## 9. Verificaciones

In [26]:
for nombre, d in final.items():
    assert d.isna().sum().sum() == 0, f"{nombre} tiene nulos"
    assert list(d.columns) == COLUMNAS_SALIDA, f"{nombre} tiene columnas inesperadas"

assert final["train"]["pickup_datetime"].max() < INICIO_VAL
assert final["val"]["pickup_datetime"].min() >= INICIO_VAL
assert final["val"]["pickup_datetime"].max() < INICIO_TEST
assert final["test"]["pickup_datetime"].min() >= INICIO_TEST
assert final["test"]["pickup_datetime"].max() < FIN_MES

total_final = sum(len(d) for d in final.values())
resumen = pd.DataFrame({
    nombre: {
        "filas": len(d),
        "%_del_limpio": len(d) / total_final * 100,
        "desde": d["pickup_datetime"].min(),
        "hasta": d["pickup_datetime"].max(),
        "fare_media": d["fare_amount"].mean(),
        "duracion_media_min": d["trip_duration_min"].mean(),
    } for nombre, d in final.items()
}).T
print(f"Crudo: {n_raw:,} → Limpio: {total_final:,} ({total_final / n_raw * 100:.2f}% conservado)")
resumen

Crudo: 3,588,295 → Limpio: 3,411,905 (95.08% conservado)


,filas,%_del_limpio,desde,hasta,fare_media,duracion_media_min
train,2393191,70.142,2022-05-01 00:00:00,2022-05-21 23:59:59,14.412,16.252
val,546762,16.025,2022-05-22 00:00:00,2022-05-26 23:59:58,14.793,16.866
test,471952,13.833,2022-05-27 00:00:00,2022-05-31 23:59:59,14.581,15.156


### Baseline de referencia

Predecir la **mediana de train** para cada target. Cualquier modelo debe tener un MAE menor que este.

In [27]:
baseline = {}
for target in COLUMNAS_TARGET:
    mediana_train = final["train"][target].median()
    baseline[target] = {
        "mediana_train": mediana_train,
        **{f"MAE_{s}": (final[s][target] - mediana_train).abs().mean() for s in ["val", "test"]},
    }
pd.DataFrame(baseline).T

,mediana_train,MAE_val,MAE_test
fare_amount,10.500,7.464,7.523
trip_duration_min,12.780,8.869,7.903


## 10. Guardado

In [28]:
COMPRIMIR = True          # True → .csv.gz (cabe en GitHub); False → .csv normal (train > 100 MB)
FILAS_MUESTRA = 10_000    # muestra en CSV normal para abrir en Excel / VS Code

compresion = {"method": "gzip", "compresslevel": 6} if COMPRIMIR else None
extension = ".csv.gz" if COMPRIMIR else ".csv"

for nombre, d in final.items():
    ruta = OUT_DIR / f"{nombre}{extension}"
    d.to_csv(ruta, index=False, float_format="%.2f", compression=compresion)
    mb = ruta.stat().st_size / 1e6
    print(f"{ruta.relative_to(ROOT)}: {mb:,.1f} MB | {len(d):,} filas")
    if mb > 100:
        print("   ⚠️ supera 100 MB: GitHub lo rechazará")

muestra = final["train"].sample(FILAS_MUESTRA, random_state=42).sort_values("pickup_datetime")
muestra.to_csv(OUT_DIR / "muestra_train.csv", index=False, float_format="%.2f")
print(f"data/processed/muestra_train.csv: {FILAS_MUESTRA:,} filas")

log_total = pd.concat([log_fijas, pd.DataFrame(log_topes)], ignore_index=True)
log_total.to_csv(OUT_DIR / "limpieza_log.csv", index=False)
print("data/processed/limpieza_log.csv guardado")

data\processed\train.csv.gz: 27.6 MB | 2,393,191 filas
data\processed\val.csv.gz: 6.3 MB | 546,762 filas
data\processed\test.csv.gz: 5.5 MB | 471,952 filas
data/processed/muestra_train.csv: 10,000 filas
data/processed/limpieza_log.csv guardado


## Cómo usar la data limpia (para el resto del equipo)

```python
import pandas as pd
train = pd.read_csv("data/processed/train.csv.gz", parse_dates=["pickup_datetime"])
val   = pd.read_csv("data/processed/val.csv.gz",   parse_dates=["pickup_datetime"])
test  = pd.read_csv("data/processed/test.csv.gz",  parse_dates=["pickup_datetime"])

features = ["trip_distance", "PULocationID", "DOLocationID", "pickup_hour", "pickup_dow",
            "is_airport_trip", "morning_rush", "night_rush", "weekend_rush"]
X_train = train[features]
y_fare = train["fare_amount"]          # USD
y_dur  = train["trip_duration_min"]    # minutos
```

- `pickup_datetime` se deja solo para trazabilidad; no es feature.
- `muestra_train.csv` (10,000 filas) es solo para mirar la data en Excel; no se usa para entrenar.
- Encoding / escalado va dentro del pipeline de cada modelo y se ajusta **solo con train**.